# 健康人 MSN 构建与拓扑提取

本 notebook 针对 **健康对照（control 组）**：

- 从 `data/health_feature_matrices/` 读取每个被试的特征矩阵（行=脑区/体积指标，列=特征）。
- 使用 `msn_pipeline.msn.build_msn` 为每个被试构建 MSN（参数与主 `run_msn_pipeline` 以及患者 MSN notebook 完全一致，例如 `metric="pearson"`, `target_density=0.2`）。
- 使用 `msn_pipeline.features.calculate_graph_metrics` 提取全局拓扑指标（Eg, Eloc, Cp, Lp，σ 可选）。
- 使用 `msn_pipeline.features.compute_nodal_topology` 提取节点级拓扑指标（degree_centrality、betweenness、clustering、eigenvector、pagerank）。
- 将结果导出为：
  - 每个健康人的 MSN 矩阵 CSV（**Pearson 相关经 Fisher Z 变换后保存**），目录为 `results/health_msn_matrices/`，文件名约定为 `{subject_id}_msn.csv`。
  - 汇总的全局拓扑指标表 `results/health_subject_topology.csv`（包含 `subject_id`, `group`=0 以及各全局拓扑指标列）。
  - （可选）每个健康人的节点级拓扑指标表，目录为 `results/health_nodal_topology/`（供后续 GLM/ANCOVA 使用）。

> 本 notebook 的目标是为后续 **健康 vs 患者** 以及 **患者亚型间** 的拓扑/NBS/GLM 分析提供干净一致的健康人 MSN 及拓扑特征表。

In [ ]:
# 1. 环境和路径

import sys
from pathlib import Path

# 自动识别项目根：当前目录或上一级（在 notebook/ 下运行时）
ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "msn_pipeline").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

print("项目根:", ROOT)
print("msn_pipeline 可导入:", (ROOT / "src" / "msn_pipeline").exists())


In [ ]:
# 2. 参数配置

import pandas as pd
import numpy as np

from msn_pipeline.data import DataPaths, load_subject_features
from msn_pipeline.msn import build_msn
from msn_pipeline.features import calculate_graph_metrics, compute_nodal_topology
from msn_pipeline.pipelines import build_group_msns

# 基础路径（与项目结构保持一致）
DATA_ROOT = ROOT / "data"
RESULTS_ROOT = ROOT / "results"

# MSN 参数，应与患者/主管线保持一致
MSN_METRIC = "pearson"
TARGET_DENSITY = 0.2

# 是否计算小世界 σ（较慢），可按需打开
COMPUTE_SIGMA = False
# 是否为每个被试保存节点级拓扑指标
SAVE_NODAL = False

# 健康人 MSN / 节点拓扑输出目录（与顶部 markdown 约定一致）
HEALTH_MSN_DIR = RESULTS_ROOT / "health_msn_matrices"
HEALTH_NODAL_DIR = RESULTS_ROOT / "health_nodal_topology"
HEALTH_MSN_DIR.mkdir(parents=True, exist_ok=True)
HEALTH_NODAL_DIR.mkdir(parents=True, exist_ok=True)

# 断点续算：进度保存路径（存在则下次运行会跳过已完成的被试）
PROGRESS_CSV = RESULTS_ROOT / "health_msn_progress.csv"


In [ ]:
# 3. 加载健康人特征矩阵

# 健康人特征矩阵目录由 DataPaths 统一管理
paths = DataPaths(root=DATA_ROOT)
health_dir = paths.control_csv_dir()
print("健康人特征矩阵目录:", health_dir)

subjects = load_subject_features(health_dir)
print("健康被试数量:", len(subjects))
if subjects:
    sid, df = next(iter(subjects.items()))
    print("示例 subject_id:", sid, "特征表形状:", df.shape)




本节对每个健康被试：

- 使用 `build_msn` 构建个体 MSN 网络（相似度方法为 **Pearson 相关**，与患者管线一致）。
- 对得到的 Pearson 相关矩阵做 **Fisher Z 变换**（z = arctanh(r)），再保存为 CSV，便于后续组间/NBS 等统计（Z 近似正态、方差稳定）。
- 将 Fisher Z 矩阵保存到 `results/health_msn_matrices/`（行列均为脑区名称）。
- 同时计算全局拓扑指标，并（可选）计算/保存节点级拓扑指标，供下一节汇总为特征表。

**说明**：Eg/Eloc/Cp/Lp（及后续单独补算的 σ）均基于 **Pearson 相关** 构建的图 G 计算；Fisher Z 仅用于**保存矩阵到 CSV**，不参与建图与拓扑计算。若不想在主循环中算 σ（较慢），保持 `COMPUTE_SIGMA = False`，跑完第 5 节后可用下方「单独计算小世界系数」单元从已保存的 Z 矩阵补算 σ。

**断点续算**：进度会写入 `results/health_msn_progress.csv`（每完成一人保存一次）。中途停止后重新运行第 4 节代码单元即可从未完成的被试接着算；全部跑完后可删除该进度文件或保留作备份。


In [ ]:
# 4. 为每个健康人构建 MSN 并保存
# 重构后：调用 msn_pipeline.pipelines.build_group_msns 承担核心逻辑

health_topology_df = build_group_msns(
    group="health",
    data_root=DATA_ROOT,
    results_root=RESULTS_ROOT,
    metric=MSN_METRIC,
    target_density=TARGET_DENSITY,
    compute_sigma=COMPUTE_SIGMA,
    save_nodal=SAVE_NODAL,
    progress_csv=PROGRESS_CSV,
)

print("build_msn 完成，健康被试数量:", len(health_topology_df))
if not health_topology_df.empty:
    print("示例 subject_id:", health_topology_df["subject_id"].iloc[0])

In [ ]:
# 将 global_rows 汇总为 DataFrame，并保存为健康人全局拓扑主表

health_topology_df = pd.DataFrame(global_rows)
if health_topology_df.empty:
    raise RuntimeError("global_rows 为空，检查上一节是否成功运行并收集到拓扑指标。")

# 添加 group 列：健康人记为 0
health_topology_df["group"] = 0


In [ ]:
# 单独计算小世界系数 σ（可选；主循环 COMPUTE_SIGMA=False 时在此补算）
# 从已保存的 Fisher Z 矩阵 CSV 反推 Pearson r，再建图并算 σ

from msn_pipeline.msn import create_network

if "σ" not in health_topology_df.columns:
    print("开始单独计算小世界系数 σ ...")
    sigma_list = []
    for row in global_rows:
        sid = row["subject_id"]
        msn_path = HEALTH_MSN_DIR / f"{sid}_msn.csv"
        if not msn_path.exists():
            sigma_list.append(np.nan)
            continue
        z_df = pd.read_csv(msn_path, index_col=0)
        z_matrix = z_df.values.astype(float)
        r_matrix = np.tanh(z_matrix)  # Fisher Z 逆变换 -> Pearson r
        regions = list(z_df.index)
        G = create_network(regions, r_matrix, target_density=TARGET_DENSITY)
        sigma = calculate_graph_metrics(G, metrics=["σ"]).get("σ", np.nan)
        sigma_list.append(sigma)
    health_topology_df["σ"] = sigma_list
    print("已补充添加 σ 到 health_topology_df。")
else:
    print("health_topology_df 已包含 σ，无需重新计算。")

# 5. 汇总全局拓扑指标并导出

本节将上一节循环中得到的每个健康被试的全局拓扑指标汇总成 `DataFrame`，并：

- 添加 `group` 列，用于区分健康/患者（约定健康 `group=0`）。
- 统一列顺序为：`subject_id`, `group`, 各拓扑指标列。
- 输出为 `results/health_subject_topology.csv`，供后续 NBS、GLM/ANCOVA 分析使用。

In [ ]:

# 统一列顺序：subject_id, group, 其余拓扑指标列
cols = ["subject_id", "group"] + [
    c for c in health_topology_df.columns
    if c not in ("subject_id", "group")
]
health_topology_df = health_topology_df[cols]

# 保存到 results/health_subject_topology.csv
topology_path = RESULTS_ROOT / "health_subject_topology.csv"
topology_path.parent.mkdir(parents=True, exist_ok=True)
health_topology_df.to_csv(topology_path, index=False)
print("已保存健康人全局拓扑指标到:", topology_path)
health_topology_df.head()